# 🧠  — AI Job Description Analyzer
**LLM Engineering Course — Ed Donner**
*Author: Abdulrahman Alanani*

---

Paste any AI/ML job description and get an instant skill gap analysis against your CV.

**What this covers:**
- Prompt engineering (system prompt vs user prompt)
- Structured JSON output from LLMs
- OpenAI API call pattern
- Defensive output parsing

Run the cells **top to bottom**, one at a time.


In [ ]:
# Cell 1 — Install dependencies (run once)
# If already installed, this will just confirm versions

!pip install openai python-dotenv --quiet
print("✅ Dependencies ready")


In [ ]:
# Cell 2 — Imports & load API key from .env file

import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()  # reads OPENAI_API_KEY from your .env file in the same folder

client = OpenAI()
print("✅ OpenAI client ready")


## Step 1 — Define your CV context

This is the **system-level knowledge** about the candidate.  
In a real production app, this would be loaded from a database or file — not hardcoded.  
For now, we keep it as a string constant that gets injected into the system prompt.


In [ ]:
# Cell 3 — Your CV (injected into the system prompt as static context) 
# Change it to be yours don't use my CV :)

CV_CONTEXT = """
Name: Abdulrahman Alanani
Target Roles: Junior AI Engineer / LLM Engineer / Agentic AI Engineer / Generative AI Engineer

EXPERIENCE:
- AI/ML Systems Support Engineer (Dec 2024–Dec 2025)
  Organization: Problem Solver Org / Future of Egypt (Egyptian Army), Cairo
  * Built and deployed anomaly detection model for IT infrastructure monitoring → 15% downtime reduction
  * Automated system diagnostics with Python scripts → 20% resolution time improvement
  * 95% first-contact resolution across 200+ monthly incidents

KEY PROJECT:
- AI-Powered Autonomous Drone for Fire Detection & Extinguishing (Graduation Project — Grade: A+)
  * Trained YOLOv8 achieving mAP50 0.824, precision 0.83 on 754-image validation set
  * Full CV pipeline: Roboflow → Google Colab (Tesla T4) → real-time edge inference on Raspberry Pi 3
  * Stack: Python, YOLOv8, TensorFlow, PyTorch, OpenCV, Roboflow, Raspberry Pi, APM 2.8, C++

EDUCATION:
- B.Eng. Computer and Systems Engineering — Badr University Cairo (Graduated Jul 2024)

TECHNICAL SKILLS:
- Programming   : Python (Advanced), C++, SQL, Git
- AI / ML       : YOLOv8, TensorFlow, PyTorch, OpenCV, Deep Learning, Object Detection, Image Segmentation
- LLM & GenAI   : LLM APIs (learning), Prompt Engineering (beginner), RAG (in progress), LLMOps fundamentals
- MLOps & Deploy: Docker, CI/CD for AI, Model Deployment, Monitoring, Evaluation, Edge AI
- Cloud         : AWS SageMaker, Oracle Cloud (OCI), Google Colab, Raspberry Pi

CERTIFICATIONS:
- LLMOps Specialization — DeepLearning.AI (Sep 2025)
- Become an LLM Engineer in 8 Weeks — Ed Donner (In Progress, 2025)
- OCI 2025 AI Foundations Associate — Oracle (Sep 2025)
- AWS Educate Machine Learning Foundations — AWS (Jun 2025)
- Introducing Generative AI with AWS — Udacity (Jun 2025)
- Artificial Intelligence Diploma (72 hrs.) — ITI Cairo (Jul 2023)

LOCATION: Riyadh, Saudi Arabia. Open to remote.
LANGUAGES: Arabic (Native), English (Professional)
"""

print("✅ CV context loaded")
print(f"   {len(CV_CONTEXT)} characters")


## Step 2 — Build the system prompt

The **system prompt** sets the model's role, rules, and output format.  
Key engineering decision here: we force **JSON-only output** so the response is  
machine-parseable, not just human-readable.


In [ ]:
# Cell 4 — System prompt (model role + output format instructions) 
# You can adjusrt the system prompt as you wish for my case I made it helpful to me because I am searching for AI Engineering jobs 

system_prompt = """
You are an expert AI hiring consultant and career coach specializing in AI/ML engineering roles.
You receive a candidate's CV and a job description, then perform a precise, honest skill gap analysis.

Respond ONLY with valid JSON — no markdown fences, no explanation, no extra text.
Use exactly this structure:

{
  "role_title": "extracted job title from the JD",
  "company": "extracted company name, or 'Not specified'",
  "match_score": <integer 0-100>,
  "score_label": "one of: Strong Match / Good Match / Partial Match / Weak Match",
  "matched_skills": ["skill1", "skill2"],
  "missing_skills": ["skill1", "skill2"],
  "transferable_skills": ["skill — brief reason why it transfers"],
  "summary": "2-3 sentence honest assessment of fit for this specific role",
  "top_actions": ["action1", "action2", "action3", "action4"]
}

Field rules:
- matched_skills     : skills from the JD the candidate clearly already has
- missing_skills     : skills explicitly required that the candidate lacks
- transferable_skills: candidate strengths not listed in JD but genuinely relevant
- top_actions        : specific, prioritized steps to close the gap (not generic advice)
- Be brutally honest. Do not inflate the score.
  A junior candidate with no LLM projects is at most a Partial Match for senior LLM roles.
"""

print("✅ System prompt ready")
print(f"   {len(system_prompt)} characters")


## Step 3 — Paste the job description

Replace the text inside `sample_jd` with any real job posting.  
This is the **user prompt input** — the dynamic part that changes every run.


In [ ]:
# Cell 5 — Job description to analyze
# ✏️  Replace this with any real JD you find on LinkedIn or any job board

sample_jd = """
Job Title: Junior AI/ML Engineer

Location: Hybrid/Remote

Type: Full-time

About the Role:
We are seeking a talented Junior AI/ML Engineer to join our growing AI team and contribute to developing cutting-edge machine learning solutions. You will work closely with senior engineers to build and deploy ML models, maintain data pipelines, and help optimize existing AI systems. This role offers excellent opportunities to gain hands-on experience with real-world AI applications while being mentored by industry experts.

Key Responsibilities:
Assist in developing and implementing machine learning models and algorithms
Help maintain and optimize data preprocessing pipelines
Support the testing and validation of AI models
Contribute to model deployment and monitoring processes
Participate in data collection and cleaning activities
Assist in documenting AI/ML processes and results
Help troubleshoot issues in production ML systems
Collaborate with cross-functional teams on AI initiatives
Perks:
Comprehensive health insurance and wellness programs
Flexible work arrangements (hybrid/remote options)
Professional development and certification support
Regular mentorship from senior AI engineers
Junior AI/ML Engineer Responsibilities
Hiring a junior ai/ml engineer? Here's what you can expect them to handle:

Build and maintain ML models using popular frameworks like TensorFlow and PyTorch
Assist in data preprocessing, feature engineering, and dataset preparation
Help implement and maintain ML pipelines for model training and deployment
Support the integration of ML models into production systems
Participate in code reviews and technical documentation
Assist in monitoring model performance and implementing improvements
Collaborate with data scientists on model development and optimization
Contribute to research and implementation of new ML techniques
"""

print("✅ Job description loaded")
print(f"   {len(sample_jd)} characters")


## Step 4 — Build the messages list

This is the standard OpenAI chat format:
- `system` → sets the model's role, rules, and output schema (static — your CV lives here)
- `user` → the actual request (dynamic — the JD goes here)

> 💡 Keeping static context in `system` and dynamic input in `user` is the correct
> pattern for any LLM app where the context is fixed but the query changes per run.


In [ ]:
# Cell 6 — Assemble the messages list

user_prompt = f"""
CANDIDATE CV:
{CV_CONTEXT}

JOB DESCRIPTION:
{sample_jd}
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": user_prompt},
]

print("✅ Messages list ready")
print(f"   system : {len(system_prompt)} chars")
print(f"   user   : {len(user_prompt)} chars")
print(f"   total  : {len(messages)} messages")


## Step 5 — Call the OpenAI API

Single API call. We use `gpt-5.4-nano` — fast and cheap for experimentation.  
`raw_output` holds the model's raw text response before we parse it.


In [ ]:
# Cell 7 — Make the API call

# ─────────────────────────────────────────────────────────────────
# OPTION A — OpenAI API  (default, requires OPENAI_API_KEY in .env in the same folder as the project)
# ─────────────────────────────────────────────────────────────────

response = client.chat.completions.create(
    model="gpt-5.4-nano",   # swap to "gpt-5.4-mini" for higher quality
    messages=messages,
)

# ─────────────────────────────────────────────────────────────────
# OPTION B — Free local LLM via Ollama  (no API key needed)
#
# Step 1 — Install Ollama: https://ollama.com
# Step 2 — Pull a model in your terminal (choose one):
#   ollama pull llama3      → Meta Llama 3 8B   (~5 GB, best quality)
#   ollama pull mistral     → Mistral 7B        (~4 GB, fast + capable)
#   ollama pull phi3        → Microsoft Phi-3   (~2 GB, lightest option)
# Step 3 — Ollama starts a local server at http://localhost:11434
#   The OpenAI client points to it directly — no other code changes needed.
#
# To switch: comment out Option A above, uncomment the block below.
# ─────────────────────────────────────────────────────────────────
# from openai import OpenAI
#
# client = OpenAI(
#     base_url="http://localhost:11434/v1",  # Ollama local server
#     api_key="ollama",                      # required field — Ollama ignores the value
# )
#
# response = client.chat.completions.create(
#     model="llama3",   # must match what you pulled: ollama pull llama3
#     messages=messages,
# )
# ─────────────────────────────────────────────────────────────────

raw_output = response.choices[0].message.content

print("✅ Response received")
print(f"   Model         : {response.model}")
print(f"   Prompt tokens : {response.usage.prompt_tokens}")
print(f"   Output tokens : {response.usage.completion_tokens}")
print(f"   Total tokens  : {response.usage.total_tokens}")
print()
print("── Raw output preview ──────────────────────────")
print(raw_output[:300], "..." if len(raw_output) > 300 else "")


## Step 6 — Parse & display the result

We defensively strip any markdown fences the model might add,  
then parse the JSON and print a clean, readable report.


In [ ]:
# Cell 8 — Parse JSON and print the full analysis report

try:
    # Defensive strip: remove ```json or ``` fences if model adds them
    clean = raw_output.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    result = json.loads(clean)

    print("=" * 60)
    print(f"  ROLE    : {result['role_title']}")
    print(f"  COMPANY : {result['company']}")
    print(f"  SCORE   : {result['match_score']}% — {result['score_label']}")
    print("=" * 60)

    print("\n📋 SUMMARY")
    print(f"  {result['summary']}")

    print("\n✅ SKILLS YOU ALREADY HAVE")
    for s in result["matched_skills"]:
        print(f"  • {s}")

    print("\n❌ GAPS TO CLOSE")
    for s in result["missing_skills"]:
        print(f"  • {s}")

    print("\n🔄 TRANSFERABLE STRENGTHS")
    for s in result["transferable_skills"]:
        print(f"  • {s}")

    print("\n🎯 YOUR ACTION PLAN")
    for i, action in enumerate(result["top_actions"], 1):
        print(f"  {i}. {action}")

    print("\n" + "=" * 60)

except json.JSONDecodeError as e:
    print("❌ JSON parsing failed. Raw model output:")
    print(raw_output)
    raise e


## Bonus — Inspect the raw result dictionary

The `result` variable is a plain Python dict.  
You can access any field directly for further processing.


In [ ]:
# Cell 9 — Inspect individual fields (optional)

print("Match score  :", result["match_score"])
print("Score label  :", result["score_label"])
print("Missing skills:", result["missing_skills"])

# The full dict if you want to explore everything
# result
